## NAMD Analysis, Fitting, Bootstrap, and Data Export

This is the main data-processing cell for the NAMD analysis. It reads the raw output from the non-adiabatic molecular dynamics (NAMD) simulations, performs statistical analysis, and exports the processed data for subsequent plotting. **No figures are generated in this cell.**

### Purpose

The primary goal is to extract the ground-state (S₀) population recovery dynamics from the NAMD simulations, fit each trajectory batch to a stretched exponential model, and build a statistically robust ensemble. This process allows for the calculation of non-radiative lifetimes and photoluminescence quantum yields (PLQYs).

### Workflow

The cell executes the following steps for every combination of **structure** (e.g., Adamantane, 1-Methyl-adamantane) and **method** (FSSH, FSSH2, IDA, MSDM):

1.  **Data Loading:**
    - Reads multiple HDF5 files (`mem_data.hdf`), each corresponding to a "batch" of trajectories.
    - Extracts the time array (`time/data`) and the adiabatic state populations (`sh_pop_adi/data`).
    - Isolates the ground-state population (column 0) for analysis.

2.  **Per-Batch Fitting:**
    - Fits the S₀ population curve of each batch to a **stretched exponential function**: $P_{S_0}(t) = 1 - \exp[-(t/\tau)^\beta]$.
    - This function captures the non-exponential, dispersive kinetics often observed in condensed-phase non-radiative relaxation.
    - Calculates the coefficient of determination ($R^2$) for each fit to assess its quality.

3.  **Batch Selection:**
    - Applies a pre-defined **$R^2$ threshold** (specific to each method) to filter out poorly fitted batches.
    - Only batches that meet or exceed this threshold are considered "accepted" and are used to construct the final ensemble.

4.  **Ensemble Construction & Final Fit:**
    - Interpolates all accepted batch curves onto a common time grid.
    - Computes the **ensemble average** (mean) and standard deviation of these curves.
    - Fits this final ensemble-averaged curve to obtain the definitive non-radiative lifetime ($\tau$) and stretch exponent ($\beta$) for that specific structure-method combination.

5.  **Bootstrapping & Uncertainty Quantification:**
    - Performs a **bootstrap analysis** on the ensemble of accepted batches (e.g., 1000 resampling iterations).
    - This provides robust estimates for the confidence intervals (CIs) of the fitted parameters ($\tau$ and $\beta$) and the final PLQY.

6.  **PLQY Calculation:**
    - Calculates the Photoluminescence Quantum Yield (PLQY) using the formula: $\Phi = \frac{k_{rad}}{k_{rad} + k_{nr}}$, where the non-radiative rate ($k_{nr} = 1/\tau$) is derived from the final ensemble fit, and the radiative rate ($k_{rad}$) is taken from pre-computed structure-specific values.

7.  **Data Export:**
    - Saves all critical results to the output directory (`analysis_results_batch_average`). This includes:
        - `analysis_results.pkl`: A Python pickle file containing the full results dictionary for all analyses.
        - `batch_curves.csv`: The raw data for all individual batch fits.
        - `ensemble_curves.csv`: The data for the final ensemble-averaged curves and their fits.
        - `comparison_data.csv`: A summary table with $\tau$, $\beta$, and PLQY values for all structures and methods.
        - `summary_results.txt`: A human-readable summary of the key findings.

In [ ]:
#!/usr/bin/env python3

# ============================================================
# CELL 1
# NAMD ANALYSIS + FIT + BOOTSTRAP + DATA EXPORT
#
# This cell:
#   1. Reads all HDF5 files
#   2. Extracts S0 population = sh_pop_adi/data[:, 0]
#   3. Fits every batch
#   4. Applies R2 thresholds
#   5. Builds accepted-batch ensemble
#   6. Fits final ensemble
#   7. Performs bootstrap
#   8. Calculates PLQY
#   9. Saves everything required for plotting
#
# NO FIGURES ARE GENERATED IN THIS CELL.
# ============================================================

import os
import csv
import pickle
import warnings

import h5py
import numpy as np

from scipy.optimize import curve_fit
from libra_py import units


# ============================================================
# SETTINGS
# ============================================================

BASE_DIR = "/home/hamid/A/NAMD"

STRUCTURES = {
    "Adamantane": "adamantane",
    "1-Methyl-adamantane": "1methyl_adamantane",
    "2-Methyl-adamantane": "2methyl_adamantane",
    "3-Methyl-adamantane": "3methyl_adamantane",
}

METHODS = [
    "FSSH",
    "FSSH2",
    "IDA",
    "MSDM",
]

METHOD_PREFIX = {
    "FSSH": "FSSH0",
    "FSSH2": "FSSH20",
    "IDA": "IDA0",
    "MSDM": "MSDM0",
}

R2_THRESHOLDS = {
    "FSSH": 0.8,
    "FSSH2": 0.8,
    "IDA": 0.4,
    "MSDM": 0.4,
}

ICONDS = list(range(1, 3001, 100))

NTRAJ_PER_BATCH = 1000

CONF_LEVEL = 0.95

MAX_TIME_FS = 3000.0

N_ENSEMBLE_POINTS = 1000

BETA_MIN = 0.05
BETA_MAX = 10.0

N_BOOTSTRAP = 1000
BOOTSTRAP_SEED = 12345


# ============================================================
# OUTPUT DIRECTORY
# ============================================================

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "analysis_results_batch_average"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# ============================================================
# SAVED DATA FILES
# ============================================================

RESULTS_PKL = os.path.join(
    OUTPUT_DIR,
    "analysis_results.pkl"
)

BATCH_CSV = os.path.join(
    OUTPUT_DIR,
    "batch_curves.csv"
)

ENSEMBLE_CSV = os.path.join(
    OUTPUT_DIR,
    "ensemble_curves.csv"
)

COMPARISON_CSV = os.path.join(
    OUTPUT_DIR,
    "comparison_data.csv"
)

SUMMARY_TXT = os.path.join(
    OUTPUT_DIR,
    "summary_results.txt"
)


# ============================================================
# EXPERIMENTAL DATA
# ============================================================

EXPERIMENTAL_TAU_PS = {
    "Adamantane": 7.46,
    "1-Methyl-adamantane": 16.3,
    "2-Methyl-adamantane": 38.3,
    "3-Methyl-adamantane": 106.0,
}


# Structure-level calculated radiative lifetimes.
# These are NOT method-specific NAMD quantities.

RADIATIVE_TAU_COMPUTED_NS = {
    "Adamantane": 162.0,
    "1-Methyl-adamantane": 80.4,
    "2-Methyl-adamantane": 41.8,
    "3-Methyl-adamantane": 25.1,
}

RADIATIVE_TAU_COMPUTED_ERR_NS = {
    "Adamantane": 4.3,
    "1-Methyl-adamantane": 1.6,
    "2-Methyl-adamantane": 0.8,
    "3-Methyl-adamantane": 0.6,
}


# Experimental radiative lifetime:
# tau_rad = 1 / Gamma_rad

RADIATIVE_TAU_EXPERIMENTAL_NS = {
    "Adamantane": 0.74,
    "1-Methyl-adamantane": 0.94,
    "2-Methyl-adamantane": 1.09,
    "3-Methyl-adamantane": 1.82,
}


EXPERIMENTAL_PLQY_PERCENT = {
    "Adamantane": 1.01,
    "1-Methyl-adamantane": 1.73,
    "2-Methyl-adamantane": 3.51,
    "3-Methyl-adamantane": 5.82,
}


# Rander et al.:
# relative 3s quantum yield
# NOT oscillator strength

RELATIVE_QUANTUM_YIELD_3S = {
    "Adamantane": 0.010,
    "1-Methyl-adamantane": 0.017,
    "2-Methyl-adamantane": 0.034,
    "3-Methyl-adamantane": 0.055,
}


# ============================================================
# STRETCHED EXPONENTIAL
#
# S0 population:
#
# P_S0(t) = 1 - exp[-(t/tau)^beta]
#
# tau is in fs here.
# ============================================================

def stretched_exp(t, tau, beta):

    t = np.asarray(
        t,
        dtype=float
    )

    tau = max(
        float(tau),
        1e-12
    )

    beta = float(beta)

    return (
        1.0
        -
        np.exp(
            -np.power(
                np.maximum(t, 0.0) / tau,
                beta
            )
        )
    )


# ============================================================
# X/Y MATCHING
# ============================================================

def match_xy(x, y):

    x = np.asarray(
        x,
        dtype=float
    ).ravel()

    y = np.asarray(
        y,
        dtype=float
    ).ravel()

    n = min(
        len(x),
        len(y)
    )

    if n < 2:
        return None, None

    x = x[:n]
    y = y[:n]

    mask = (
        np.isfinite(x)
        &
        np.isfinite(y)
    )

    x = x[mask]
    y = y[mask]

    if len(x) < 2:
        return None, None

    return x, y


# ============================================================
# CLEAN CURVE
# ============================================================

def clean_curve(time_fs, population):

    time_fs, population = match_xy(
        time_fs,
        population
    )

    if time_fs is None:
        raise ValueError(
            "Invalid x/y arrays"
        )

    mask = (
        (time_fs >= 0.0)
        &
        (time_fs <= MAX_TIME_FS)
    )

    time_fs = time_fs[mask]
    population = population[mask]

    if len(time_fs) < 10:
        raise ValueError(
            "Too few valid points"
        )

    order = np.argsort(
        time_fs
    )

    time_fs = time_fs[order]
    population = population[order]

    time_unique, indices = np.unique(
        time_fs,
        return_index=True
    )

    population_unique = population[
        indices
    ]

    if len(time_unique) < 10:
        raise ValueError(
            "Too few unique time points"
        )

    return (
        time_unique,
        population_unique
    )


# ============================================================
# BUILD HDF5 PATH
# ============================================================

def build_batch_filename(
    structure_folder,
    method,
    icond
):

    prefix = METHOD_PREFIX[
        method
    ]

    return os.path.join(
        BASE_DIR,
        structure_folder,
        method,
        f"{prefix}_icond_{icond}",
        "mem_data.hdf"
    )


# ============================================================
# LOAD ONE BATCH
# ============================================================

def load_batch(filename):

    with h5py.File(
        filename,
        "r"
    ) as F:

        if "time/data" not in F:
            raise KeyError(
                "Missing time/data"
            )

        if "sh_pop_adi/data" not in F:
            raise KeyError(
                "Missing sh_pop_adi/data"
            )

        time_fs = np.asarray(
            F["time/data"][:],
            dtype=float
        )

        population_all = np.asarray(
            F["sh_pop_adi/data"][:],
            dtype=float
        )

    time_fs = (
        time_fs
        *
        units.au2fs
    )

    if population_all.ndim != 2:
        raise ValueError(
            f"Unexpected population shape: "
            f"{population_all.shape}"
        )

    n = min(
        len(time_fs),
        population_all.shape[0]
    )

    time_fs = time_fs[:n]
    population_all = population_all[:n, :]

    if population_all.shape[1] < 1:
        raise ValueError(
            "No electronic states"
        )

    # IMPORTANT:
    # Column 0 is S0.
    #
    # Do NOT average over electronic states.

    s0_population = population_all[:, 0]

    return {
        "time_fs": time_fs,
        "population": s0_population,
        "nstates": population_all.shape[1],
    }


# ============================================================
# FIT ONE BATCH
# ============================================================

def fit_population_curve(
    time_fs,
    population,
    r2_min=None
):

    try:

        time_fs, population = clean_curve(
            time_fs,
            population
        )

    except Exception:

        return None

    ss_tot = np.sum(
        (
            population
            -
            np.mean(population)
        ) ** 2
    )

    if (
        not np.isfinite(ss_tot)
        or
        ss_tot < 1e-14
    ):

        return {
            "time_fs": time_fs,
            "population": population,
            "tau_fs": np.nan,
            "tau_ps": np.nan,
            "beta": np.nan,
            "r2": np.nan,
            "fitted": np.full_like(
                population,
                np.nan
            ),
            "accepted": False,
        }

    # Estimate tau near P = 1-exp(-1)

    target = (
        1.0
        -
        np.exp(-1.0)
    )

    idx = np.argmin(
        np.abs(
            population
            -
            target
        )
    )

    tau_guess = float(
        time_fs[idx]
    )

    if (
        not np.isfinite(tau_guess)
        or
        tau_guess <= 0
    ):

        positive = time_fs[
            time_fs > 0
        ]

        if len(positive):

            tau_guess = float(
                np.median(positive)
            )

        else:

            tau_guess = 1.0

    try:

        with warnings.catch_warnings():

            warnings.simplefilter(
                "ignore"
            )

            popt, _ = curve_fit(

                stretched_exp,

                time_fs,

                population,

                p0=[
                    tau_guess,
                    1.0
                ],

                bounds=(

                    [
                        1e-12,
                        BETA_MIN
                    ],

                    [
                        max(
                            300000.0,
                            MAX_TIME_FS * 100.0
                        ),
                        BETA_MAX
                    ]
                ),

                maxfev=100000
            )

    except Exception:

        return None

    tau_fs = float(
        popt[0]
    )

    beta = float(
        popt[1]
    )

    fitted = stretched_exp(
        time_fs,
        tau_fs,
        beta
    )

    ss_res = np.sum(
        (
            population
            -
            fitted
        ) ** 2
    )

    r2 = (
        1.0
        -
        ss_res / ss_tot
    )

    accepted = (
        np.isfinite(r2)
        and
        (
            r2_min is None
            or
            r2 >= r2_min
        )
    )

    return {
        "time_fs": time_fs,
        "population": population,
        "tau_fs": tau_fs,
        "tau_ps": tau_fs / 1000.0,
        "beta": beta,
        "r2": r2,
        "fitted": fitted,
        "accepted": accepted,
    }


# ============================================================
# LOAD AND FIT ALL BATCHES
# ============================================================

def load_batches(
    structure_name,
    structure_folder,
    method
):

    threshold = R2_THRESHOLDS[
        method
    ]

    accepted_batches = []
    rejected_batches = []

    diagnostics = {
        "files": 0,
        "loaded": 0,
        "accepted": 0,
        "rejected": 0,
        "fit_failed": 0,
        "missing": 0,
        "errors": [],
    }

    for icond in ICONDS:

        filename = build_batch_filename(
            structure_folder,
            method,
            icond
        )

        if not os.path.isfile(
            filename
        ):

            diagnostics[
                "missing"
            ] += 1

            continue

        diagnostics[
            "files"
        ] += 1

        try:

            batch = load_batch(
                filename
            )

            diagnostics[
                "loaded"
            ] += 1

        except Exception as exc:

            diagnostics[
                "fit_failed"
            ] += 1

            if len(
                diagnostics["errors"]
            ) < 3:

                diagnostics[
                    "errors"
                ].append(
                    (
                        icond,
                        str(exc)
                    )
                )

            continue

        fit = fit_population_curve(
            batch["time_fs"],
            batch["population"],
            r2_min=threshold
        )

        if fit is None:

            diagnostics[
                "fit_failed"
            ] += 1

            continue

        result = {
            **batch,
            **fit,
            "icond": icond,
            "filename": filename,
        }

        if fit["accepted"]:

            accepted_batches.append(
                result
            )

            diagnostics[
                "accepted"
            ] += 1

        else:

            rejected_batches.append(
                result
            )

            diagnostics[
                "rejected"
            ] += 1

    print(
        f"{structure_name:<24s} | "
        f"{method:<5s} | "
        f"loaded={diagnostics['loaded']:2d} "
        f"accepted={diagnostics['accepted']:2d} "
        f"rejected={diagnostics['rejected']:2d}"
    )

    return (
        accepted_batches,
        rejected_batches,
        diagnostics
    )


# ============================================================
# COMMON TIME GRID
# ============================================================

def make_time_grid():

    return np.linspace(
        0.0,
        MAX_TIME_FS,
        N_ENSEMBLE_POINTS
    )


# ============================================================
# INTERPOLATE ONE BATCH
# ============================================================

def interpolate_one_batch(
    batch,
    time_grid
):

    t, y = match_xy(
        batch["time_fs"],
        batch["population"]
    )

    if t is None:
        return None

    try:

        t, y = clean_curve(
            t,
            y
        )

    except Exception:

        return None

    result = np.full(
        len(time_grid),
        np.nan,
        dtype=float
    )

    valid = (
        (time_grid >= t[0])
        &
        (time_grid <= t[-1])
    )

    if np.any(valid):

        result[valid] = np.interp(
            time_grid[valid],
            t,
            y
        )

    return result


# ============================================================
# INTERPOLATE BATCHES
# ============================================================

def interpolate_batches(
    batches,
    time_grid
):

    matrix = []

    for batch in batches:

        yi = interpolate_one_batch(
            batch,
            time_grid
        )

        if yi is not None:

            matrix.append(yi)

    if len(matrix) == 0:
        return None

    return np.asarray(
        matrix,
        dtype=float
    )


# ============================================================
# BATCH ENSEMBLE
# ============================================================

def build_batch_ensemble(
    accepted_batches,
    time_grid
):

    matrix = interpolate_batches(
        accepted_batches,
        time_grid
    )

    if matrix is None:
        return None

    with warnings.catch_warnings():

        warnings.simplefilter(
            "ignore"
        )

        mean_curve = np.nanmean(
            matrix,
            axis=0
        )

        std_curve = np.nanstd(
            matrix,
            axis=0,
            ddof=1
        )

    std_curve[
        ~np.isfinite(std_curve)
    ] = 0.0

    return {
        "matrix": matrix,
        "mean": mean_curve,
        "std": std_curve,
        "time_fs": time_grid,
    }


# ============================================================
# FINAL ENSEMBLE FIT
# ============================================================

def fit_final_ensemble(
    ensemble
):

    if ensemble is None:
        return None

    return fit_population_curve(
        ensemble["time_fs"],
        ensemble["mean"],
        r2_min=None
    )


# ============================================================
# BOOTSTRAP ACCEPTED BATCHES
# ============================================================

def bootstrap_batches(
    ensemble
):

    if ensemble is None:
        return None

    matrix = ensemble[
        "matrix"
    ]

    n_batches = matrix.shape[0]

    if n_batches < 2:
        return None

    rng = np.random.default_rng(
        BOOTSTRAP_SEED
    )

    tau_samples = []
    beta_samples = []

    for _ in range(
        N_BOOTSTRAP
    ):

        indices = rng.integers(
            0,
            n_batches,
            size=n_batches
        )

        sample = matrix[
            indices,
            :
        ]

        with warnings.catch_warnings():

            warnings.simplefilter(
                "ignore"
            )

            mean_curve = np.nanmean(
                sample,
                axis=0
            )

        fit = fit_population_curve(
            ensemble["time_fs"],
            mean_curve,
            r2_min=None
        )

        if fit is None:
            continue

        if (
            np.isfinite(
                fit["tau_ps"]
            )
            and
            np.isfinite(
                fit["beta"]
            )
        ):

            tau_samples.append(
                fit["tau_ps"]
            )

            beta_samples.append(
                fit["beta"]
            )

    tau_samples = np.asarray(
        tau_samples,
        dtype=float
    )

    beta_samples = np.asarray(
        beta_samples,
        dtype=float
    )

    if len(tau_samples) < 10:
        return None

    alpha = (
        1.0
        -
        CONF_LEVEL
    )

    low = 100.0 * alpha / 2.0

    high = 100.0 * (
        1.0
        -
        alpha / 2.0
    )

    tau_ci = np.percentile(
        tau_samples,
        [low, high]
    )

    beta_ci = np.percentile(
        beta_samples,
        [low, high]
    )

    t_grid = ensemble[
        "time_fs"
    ]

    curves = np.asarray(
        [
            stretched_exp(
                t_grid,
                tau * 1000.0,
                beta
            )
            for tau, beta in zip(
                tau_samples,
                beta_samples
            )
        ],
        dtype=float
    )

    fit_lower = np.percentile(
        curves,
        low,
        axis=0
    )

    fit_upper = np.percentile(
        curves,
        high,
        axis=0
    )

    return {
        "tau_samples": tau_samples,
        "beta_samples": beta_samples,
        "tau_ci": tau_ci,
        "beta_ci": beta_ci,
        "fit_lower": fit_lower,
        "fit_upper": fit_upper,
        "n_success": len(
            tau_samples
        ),
    }


# ============================================================
# PLQY
# ============================================================

def calculate_plqy_percent(
    tau_nr_ps,
    tau_r_ns
):

    tau_r_ps = (
        tau_r_ns
        *
        1000.0
    )

    return (
        100.0
        *
        tau_nr_ps
        /
        (
            tau_nr_ps
            +
            tau_r_ps
        )
    )


def bootstrap_plqy(
    bootstrap_result,
    tau_r_ns
):

    if bootstrap_result is None:
        return None

    samples = np.asarray(
        [
            calculate_plqy_percent(
                tau,
                tau_r_ns
            )
            for tau in
            bootstrap_result[
                "tau_samples"
            ]
        ],
        dtype=float
    )

    low = (
        100.0
        *
        (1.0 - CONF_LEVEL)
        /
        2.0
    )

    high = (
        100.0
        *
        (
            1.0
            -
            (1.0 - CONF_LEVEL) / 2.0
        )
    )

    return {
        "samples": samples,
        "ci": np.percentile(
            samples,
            [low, high]
        )
    }


# ============================================================
# ANALYZE ONE STRUCTURE / METHOD
# ============================================================

def analyze_one(
    structure_name,
    structure_folder,
    method
):

    (
        accepted,
        rejected,
        diagnostics
    ) = load_batches(
        structure_name,
        structure_folder,
        method
    )

    result = {
        "structure": structure_name,
        "method": method,
        "accepted_batches": accepted,
        "rejected_batches": rejected,
        "diagnostics": diagnostics,
        "ensemble": None,
        "final_fit": None,
        "bootstrap": None,
        "plqy": None,
    }

    if len(accepted) == 0:
        return result

    time_grid = make_time_grid()

    ensemble = build_batch_ensemble(
        accepted,
        time_grid
    )

    result[
        "ensemble"
    ] = ensemble

    if ensemble is None:
        return result

    final_fit = fit_final_ensemble(
        ensemble
    )

    result[
        "final_fit"
    ] = final_fit

    bootstrap = bootstrap_batches(
        ensemble
    )

    result[
        "bootstrap"
    ] = bootstrap

    if (
        bootstrap is not None
        and
        final_fit is not None
    ):

        tau_r = (
            RADIATIVE_TAU_COMPUTED_NS[
                structure_name
            ]
        )

        result[
            "plqy"
        ] = bootstrap_plqy(
            bootstrap,
            tau_r
        )

    return result


# ============================================================
# RUN ALL CALCULATIONS
# ============================================================

all_results = {}

for structure_name, structure_folder in (
    STRUCTURES.items()
):

    for method in METHODS:

        all_results[
            (
                structure_name,
                method
            )
        ] = analyze_one(
            structure_name,
            structure_folder,
            method
        )


# ============================================================
# ADD GLOBAL METADATA
# ============================================================

saved_data = {

    "all_results":
        all_results,

    "structures":
        STRUCTURES,

    "methods":
        METHODS,

    "method_prefix":
        METHOD_PREFIX,

    "r2_thresholds":
        R2_THRESHOLDS,

    "iconds":
        ICONDS,

    "ntraj_per_batch":
        NTRAJ_PER_BATCH,

    "confidence_level":
        CONF_LEVEL,

    "max_time_fs":
        MAX_TIME_FS,

    "n_ensemble_points":
        N_ENSEMBLE_POINTS,

    "n_bootstrap":
        N_BOOTSTRAP,

    "bootstrap_seed":
        BOOTSTRAP_SEED,

    "experimental_tau_ps":
        EXPERIMENTAL_TAU_PS,

    "radiative_tau_computed_ns":
        RADIATIVE_TAU_COMPUTED_NS,

    "radiative_tau_computed_err_ns":
        RADIATIVE_TAU_COMPUTED_ERR_NS,

    "radiative_tau_experimental_ns":
        RADIATIVE_TAU_EXPERIMENTAL_NS,

    "experimental_plqy_percent":
        EXPERIMENTAL_PLQY_PERCENT,

    "relative_quantum_yield_3s":
        RELATIVE_QUANTUM_YIELD_3S,
}


# ============================================================
# SAVE PICKLE
# ============================================================

with open(
    RESULTS_PKL,
    "wb"
) as f:

    pickle.dump(
        saved_data,
        f,
        protocol=pickle.HIGHEST_PROTOCOL
    )


# ============================================================
# SAVE BATCH CURVES
#
# This contains the digital data behind the QC curves.
# ============================================================

with open(
    BATCH_CSV,
    "w",
    newline=""
) as f:

    writer = csv.writer(f)

    writer.writerow([
        "structure",
        "method",
        "icond",
        "status",
        "time_fs",
        "s0_population",
        "tau_ps",
        "beta",
        "r2",
    ])

    for (
        structure_name,
        method
    ), result in all_results.items():

        for status, batches in [
            (
                "accepted",
                result[
                    "accepted_batches"
                ]
            ),
            (
                "rejected",
                result[
                    "rejected_batches"
                ]
            ),
        ]:

            for batch in batches:

                for t, y in zip(
                    batch["time_fs"],
                    batch["population"]
                ):

                    writer.writerow([
                        structure_name,
                        method,
                        batch["icond"],
                        status,
                        t,
                        y,
                        batch["tau_ps"],
                        batch["beta"],
                        batch["r2"],
                    ])


# ============================================================
# SAVE ENSEMBLE CURVES
#
# This contains the digital data used by the main 2x2 plots.
# ============================================================

with open(
    ENSEMBLE_CSV,
    "w",
    newline=""
) as f:

    writer = csv.writer(f)

    writer.writerow([
        "structure",
        "method",
        "time_fs",
        "ensemble_mean",
        "ensemble_std",
        "fit",
        "fit_ci_lower",
        "fit_ci_upper",
    ])

    for (
        structure_name,
        method
    ), result in all_results.items():

        ensemble = result[
            "ensemble"
        ]

        final_fit = result[
            "final_fit"
        ]

        bootstrap = result[
            "bootstrap"
        ]

        if ensemble is None:
            continue

        t = ensemble[
            "time_fs"
        ]

        mean = ensemble[
            "mean"
        ]

        std = ensemble[
            "std"
        ]

        if final_fit is not None:

            fit_y = stretched_exp(
                t,
                final_fit["tau_fs"],
                final_fit["beta"]
            )

        else:

            fit_y = np.full_like(
                t,
                np.nan
            )

        if bootstrap is not None:

            lower = bootstrap[
                "fit_lower"
            ]

            upper = bootstrap[
                "fit_upper"
            ]

        else:

            lower = np.full_like(
                t,
                np.nan
            )

            upper = np.full_like(
                t,
                np.nan
            )

        for row in zip(
            t,
            mean,
            std,
            fit_y,
            lower,
            upper
        ):

            writer.writerow([
                structure_name,
                method,
                *row
            ])


# ============================================================
# SAVE COMPARISON DATA
# ============================================================

with open(
    COMPARISON_CSV,
    "w",
    newline=""
) as f:

    writer = csv.writer(f)

    writer.writerow([
        "structure",
        "method",
        "tau_ps",
        "tau_ci_low",
        "tau_ci_high",
        "beta",
        "beta_ci_low",
        "beta_ci_high",
        "plqy_percent",
        "plqy_ci_low",
        "plqy_ci_high",
        "experimental_tau_ps",
        "experimental_plqy_percent",
        "radiative_tau_computed_ns",
        "radiative_tau_computed_err_ns",
        "radiative_tau_experimental_ns",
        "relative_quantum_yield_3s",
        "n_accepted",
        "n_rejected",
    ])

    for structure_name in STRUCTURES:

        for method in METHODS:

            result = all_results[
                (
                    structure_name,
                    method
                )
            ]

            fit = result[
                "final_fit"
            ]

            boot = result[
                "bootstrap"
            ]

            plqy = result[
                "plqy"
            ]

            if fit is not None:

                tau = fit[
                    "tau_ps"
                ]

                beta = fit[
                    "beta"
                ]

            else:

                tau = np.nan
                beta = np.nan

            if boot is not None:

                tau_ci = boot[
                    "tau_ci"
                ]

                beta_ci = boot[
                    "beta_ci"
                ]

            else:

                tau_ci = [
                    np.nan,
                    np.nan
                ]

                beta_ci = [
                    np.nan,
                    np.nan
                ]

            if plqy is not None:

                plqy_value = calculate_plqy_percent(
                    tau,
                    RADIATIVE_TAU_COMPUTED_NS[
                        structure_name
                    ]
                )

                plqy_ci = plqy[
                    "ci"
                ]

            else:

                plqy_value = np.nan

                plqy_ci = [
                    np.nan,
                    np.nan
                ]

            writer.writerow([
                structure_name,
                method,

                tau,
                tau_ci[0],
                tau_ci[1],

                beta,
                beta_ci[0],
                beta_ci[1],

                plqy_value,
                plqy_ci[0],
                plqy_ci[1],

                EXPERIMENTAL_TAU_PS[
                    structure_name
                ],

                EXPERIMENTAL_PLQY_PERCENT[
                    structure_name
                ],

                RADIATIVE_TAU_COMPUTED_NS[
                    structure_name
                ],

                RADIATIVE_TAU_COMPUTED_ERR_NS[
                    structure_name
                ],

                RADIATIVE_TAU_EXPERIMENTAL_NS[
                    structure_name
                ],

                RELATIVE_QUANTUM_YIELD_3S[
                    structure_name
                ],

                len(
                    result[
                        "accepted_batches"
                    ]
                ),

                len(
                    result[
                        "rejected_batches"
                    ]
                ),
            ])


# ============================================================
# SAVE SUMMARY TEXT
# ============================================================

with open(
    SUMMARY_TXT,
    "w"
) as f:

    f.write(
        "NAMD / NBRA S0 population analysis\n"
    )

    f.write(
        "=" * 90
        +
        "\n\n"
    )

    f.write(
        "S0 = sh_pop_adi/data[:,0]\n"
    )

    f.write(
        "Each batch population is the trajectory-averaged "
        "population provided by the HDF5 data.\n"
    )

    f.write(
        "R2 is applied independently to each batch.\n"
    )

    f.write(
        "Only accepted batches enter the final ensemble.\n"
    )

    f.write(
        f"Trajectories/batch = {NTRAJ_PER_BATCH}\n"
    )

    f.write(
        f"Number of possible batches = {len(ICONDS)}\n"
    )

    f.write(
        f"Bootstrap samples = {N_BOOTSTRAP}\n\n"
    )

    for structure_name in STRUCTURES:

        for method in METHODS:

            result = all_results[
                (
                    structure_name,
                    method
                )
            ]

            fit = result[
                "final_fit"
            ]

            boot = result[
                "bootstrap"
            ]

            plqy = result[
                "plqy"
            ]

            n_acc = len(
                result[
                    "accepted_batches"
                ]
            )

            n_rej = len(
                result[
                    "rejected_batches"
                ]
            )

            f.write(
                f"{structure_name} | {method}\n"
            )

            f.write(
                f"Accepted = {n_acc}\n"
            )

            f.write(
                f"Rejected = {n_rej}\n"
            )

            if fit is not None:

                f.write(
                    f"Tau (ps) = "
                    f"{fit['tau_ps']:.8f}\n"
                )

                f.write(
                    f"Beta = "
                    f"{fit['beta']:.8f}\n"
                )

                f.write(
                    f"R2 = "
                    f"{fit['r2']:.8f}\n"
                )

            if boot is not None:

                f.write(
                    f"Tau CI95 = "
                    f"{boot['tau_ci'][0]:.8f}, "
                    f"{boot['tau_ci'][1]:.8f}\n"
                )

                f.write(
                    f"Beta CI95 = "
                    f"{boot['beta_ci'][0]:.8f}, "
                    f"{boot['beta_ci'][1]:.8f}\n"
                )

            if (
                plqy is not None
                and
                fit is not None
            ):

                value = calculate_plqy_percent(
                    fit["tau_ps"],
                    RADIATIVE_TAU_COMPUTED_NS[
                        structure_name
                    ]
                )

                f.write(
                    f"PLQY (%) = "
                    f"{value:.8f}\n"
                )

                f.write(
                    f"PLQY CI95 = "
                    f"{plqy['ci'][0]:.8f}, "
                    f"{plqy['ci'][1]:.8f}\n"
                )

            f.write("\n")


print()
print("Analysis and data export completed.")
print()
print(f"Saved results : {RESULTS_PKL}")
print(f"Batch data    : {BATCH_CSV}")
print(f"Ensemble data : {ENSEMBLE_CSV}")
print(f"Comparison    : {COMPARISON_CSV}")
print(f"Summary       : {SUMMARY_TXT}")

## Nonradiative Lifetime Comparison

This cell generates a publication-quality bar chart comparing the calculated nonradiative lifetimes ($\tau_{nr}$) for the four adamantane structures across different NAMD methods. It also includes experimental reference values for comparison.

### Purpose

The primary goal is to visualize and compare the nonradiative decay kinetics obtained from the FSSH, FSSH2, IDA, and MSDM methods. This allows for an assessment of how different theoretical treatments (e.g., inclusion of decoherence effects) affect the predicted lifetimes and how they compare with experimental data.

### Data Source

This cell **does not perform any new calculations or read raw data**. It relies entirely on the pre-processed results stored in the `analysis_results.pkl` file, which was generated by the main analysis cell.

### Methodology

1.  **Data Extraction:**
    - Loads the `analysis_results.pkl` file containing the final fitted parameters for all structure-method combinations.
    - Extracts the nonradiative lifetime ($\tau_{nr}$) and its confidence interval (CI) from the `final_fit` and `bootstrap` dictionaries for each case.

2.  **Visualization:**
    - Creates a grouped bar chart where:
        - **X-axis:** Represents the four adamantane structures.
        - **Y-axis:** Represents the nonradiative lifetime in picoseconds (ps).
    - Each method is assigned a distinct color and a slight horizontal offset to prevent bars from overlapping.
    - Experimental lifetimes are plotted as separate black diamond markers for direct comparison.
    - Error bars represent the 95% confidence interval derived from the bootstrap analysis. The visual size of these errors is capped by a pre-defined fraction (`NONRADIATIVE_ERROR_FRACTION_LIMIT`) to prevent excessively large bars from distorting the plot.

3.  **Data Table:**
    - A summary table is automatically generated and placed at the bottom of the figure.
    - This table provides the exact numerical values (mean ± error) for each calculated lifetime and the experimental reference, ensuring that the quantitative information is accessible alongside the visual representation.

4.  **Output:**
    - The final figure is saved as a high-resolution PNG file (`Nonradiative_lifetime_comparison.png`) in the specified output directory.

In [ ]:
# ============================================================
# CELL 1
# NONRADIATIVE LIFETIME COMPARISON
#
# Reads only:
#   analysis_results.pkl
#
# No HDF5 reading
# No refitting
# No new analysis
# ============================================================

import os
import pickle
import numpy as np
import matplotlib.pyplot as plt


# ============================================================
# SETTINGS
# ============================================================

OUTPUT_DIR = (
    "/home/hamid/A/NAMD/"
    "analysis_results_batch_average"
)

RESULTS_FILE = os.path.join(
    OUTPUT_DIR,
    "analysis_results.pkl"
)

FIG_DPI = 600


# ============================================================
# STRUCTURES
# ============================================================

STRUCTURES = {
    "Adamantane": "adamantane",
    "1-Methyl-adamantane": "1methyl_adamantane",
    "2-Methyl-adamantane": "2methyl_adamantane",
    "3-Methyl-adamantane": "3methyl_adamantane",
}

METHODS = [
    "MSDM",
    "IDA",
    "FSSH",
    "FSSH2",
    
]


# ============================================================
# COLORS
# ============================================================

METHOD_COLORS = {
    "FSSH": "blue",
    "FSSH2": "red",
    "IDA": "green",
    "MSDM": "purple",
}

COLOR_EXPERIMENTAL = "black"


# ============================================================
# FIGURE STYLE
# ============================================================

FIGSIZE = (16, 12)

FONT_TITLE = 28
FONT_AXIS = 24
FONT_TICKS = 20
FONT_LEGEND = 19

FONT_TABLE_HEADER = 18      # کوچک‌تر شد (قبلاً 20)
FONT_TABLE_TEXT = 16        # بزرگ‌تر شد (قبلاً 13.3)
FONT_TABLE_ROW_LABEL = 16   # جدید: اندازه فونت نام متدها در جدول

MARKER_SIZE = 14

LINE_WIDTH = 0.5

ERROR_LINE_WIDTH = 2.0
ERROR_CAPSIZE = 7
ERROR_CAPTHICK = 1.0
MARKER_EDGE_WIDTH = 1.0


# ============================================================
# HORIZONTAL OFFSETS
# ============================================================

METHOD_OFFSETS = np.linspace(
    -0.09,
    0.09,
    len(METHODS)
)

EXPERIMENTAL_OFFSET = 0.13


# ============================================================
# TABLE
# ============================================================

TABLE_BBOX_BOTTOM = 0.0
TABLE_BBOX_HEIGHT = 0.38   # کمی کاهش یافت تا جدول پایین‌تر بیاید

TABLE_TOP_DATA_Y = -45.0   # پایین‌تر آمد (قبلاً -5.0) تا خطوط وارد جدول نشوند


# ============================================================
# ERROR LIMIT
# ============================================================

NONRADIATIVE_ERROR_FRACTION_LIMIT = 0.75


# ============================================================
# LOAD SAVED DATA
# ============================================================

if not os.path.isfile(RESULTS_FILE):
    raise FileNotFoundError(
        f"Could not find:\n{RESULTS_FILE}"
    )

with open(RESULTS_FILE, "rb") as f:
    saved_data = pickle.load(f)


# ============================================================
# EXTRACT DATA
# ============================================================

if (
    isinstance(saved_data, dict)
    and "all_results" in saved_data
):
    all_results = saved_data["all_results"]
else:
    all_results = saved_data


EXPERIMENTAL_TAU_PS = (
    saved_data.get(
        "experimental_tau_ps",
        {}
    )
    if isinstance(saved_data, dict)
    else {}
)


# ============================================================
# HELPER
# ============================================================

def get_result(structure, method):

    try:
        return all_results[(structure, method)]
    except Exception:
        pass

    try:
        return all_results[structure][method]
    except Exception:
        return None


def symmetric_error_from_ci(ci):

    if ci is None:
        return np.nan

    try:
        low = float(ci[0])
        high = float(ci[1])
    except Exception:
        return np.nan

    if (
        not np.isfinite(low)
        or not np.isfinite(high)
    ):
        return np.nan

    return abs(high - low) / 2.0


def limit_error(
    value,
    error,
    fraction_limit=None,
    absolute_limit=None
):

    if (
        not np.isfinite(value)
        or not np.isfinite(error)
    ):
        return np.nan

    limited = abs(error)

    if fraction_limit is not None:
        limited = min(
            limited,
            abs(value) * fraction_limit
        )

    if absolute_limit is not None:
        limited = min(
            limited,
            absolute_limit
        )

    return max(0.0, limited)


def calculate_table_ymin(
    y_max,
    table_top_data_y
):

    return (
        table_top_data_y
        - TABLE_BBOX_HEIGHT * y_max
    ) / (
        1.0 - TABLE_BBOX_HEIGHT
    )


def position_ylabel_above_table(
    ax,
    y_min,
    y_max
):

    table_top = (
        y_min * (1.0 - TABLE_BBOX_HEIGHT)
        +
        y_max * TABLE_BBOX_HEIGHT
    )

    table_top_axes = (
        table_top - y_min
    ) / (
        y_max - y_min
    )

    y_center_axes = (
        table_top_axes + 1.0
    ) / 2.0

    ax.yaxis.set_label_coords(
        -0.08,
        y_center_axes
    )


def remove_negative_yticks(ax):

    ticks = ax.get_yticks()

    positive_ticks = [
        tick for tick in ticks
        if tick >= 0
    ]

    ax.set_yticks(positive_ticks)


def style_axes(ax):

    ax.tick_params(
        axis="both",
        which="major",
        labelsize=FONT_TICKS,
        width=1.2,
        length=6
    )

    ax.tick_params(
        axis="y",
        pad=8
    )

    remove_negative_yticks(ax)


def add_table(
    ax,
    table_data,
    row_labels,
    row_colors,
    col_labels
):

    table = ax.table(
        cellText=table_data,
        colLabels=col_labels,
        rowLabels=row_labels,
        loc="center",
        cellLoc="center",
        rowLoc="center",
        colColours=[
            "#eaeaea"
            for _ in col_labels
        ],
        bbox=[
            TABLE_BBOX_BOTTOM,
            0.0,
            1.0,
            TABLE_BBOX_HEIGHT
        ]
    )

    table.auto_set_font_size(False)

    table.set_fontsize(
        FONT_TABLE_TEXT
    )

    for (
        row,
        col
    ), cell in table.get_celld().items():

        if row == 0:

            # هدر ستون‌ها: نام ساختارها -> بدون بولد و کوچک‌تر
            cell.get_text().set_fontsize(
                FONT_TABLE_HEADER
            )

            cell.get_text().set_fontweight(
                "normal"          # بولد نباشد
            )

            cell.get_text().set_color(
                "black"
            )

            cell.set_facecolor(
                "#eaeaea"
            )

        else:

            cell.get_text().set_fontsize(
                FONT_TABLE_TEXT
            )

            if col == -1:

                # ستون نام متدها -> رنگ متن مطابق رنگ اعداد همان متد
                cell.set_facecolor(
                    "#eaeaea"
                )

                cell.get_text().set_fontweight(
                    "normal"
                )

                # رنگ نام متد = رنگ متد مربوطه
                if (
                    row - 1
                    <
                    len(row_colors)
                ):
                    cell.get_text().set_color(
                        row_colors[row - 1]
                    )
                else:
                    cell.get_text().set_color(
                        "black"
                    )

                cell.get_text().set_fontsize(
                    FONT_TABLE_ROW_LABEL
                )

            else:

                if (
                    row - 1
                    <
                    len(row_colors)
                ):

                    cell.get_text().set_color(
                        row_colors[row - 1]
                    )

        cell.set_edgecolor("black")
        cell.set_linewidth(1.0)

    return table


# ============================================================
# FIGURE
# ============================================================

fig, ax = plt.subplots(
    figsize=FIGSIZE
)

x = np.arange(
    len(STRUCTURES)
)

table_data = []
row_labels = []
row_colors = []

all_upper_values = []


# ============================================================
# METHODS
# ============================================================

for i, method in enumerate(METHODS):

    values = []
    raw_errors = []
    visual_errors = []

    for structure in STRUCTURES:

        result = get_result(
            structure,
            method
        )

        if result is None:

            values.append(np.nan)
            raw_errors.append(np.nan)
            visual_errors.append(np.nan)

            continue

        final_fit = result.get(
            "final_fit"
        )

        bootstrap = result.get(
            "bootstrap"
        )

        if final_fit is not None:

            value = float(
                final_fit.get(
                    "tau_ps",
                    np.nan
                )
            )

        else:

            value = np.nan

        if bootstrap is not None:

            ci = bootstrap.get(
                "tau_ci"
            )

        else:

            ci = None

        error = symmetric_error_from_ci(
            ci
        )

        visual_error = limit_error(
            value,
            error,
            fraction_limit=(
                NONRADIATIVE_ERROR_FRACTION_LIMIT
            )
        )

        values.append(value)
        raw_errors.append(error)
        visual_errors.append(visual_error)

        if (
            np.isfinite(value)
            and
            np.isfinite(visual_error)
        ):

            all_upper_values.append(
                value + visual_error
            )

    values = np.asarray(values)
    raw_errors = np.asarray(raw_errors)
    visual_errors = np.asarray(visual_errors)

    ax.errorbar(
        x + METHOD_OFFSETS[i],
        values,
        yerr=visual_errors,
        fmt="s-",
        color=METHOD_COLORS[method],
        markersize=MARKER_SIZE,
        linewidth=LINE_WIDTH,
        elinewidth=ERROR_LINE_WIDTH,
        capsize=ERROR_CAPSIZE,
        capthick=ERROR_CAPTHICK,
        markeredgewidth=MARKER_EDGE_WIDTH,
        label=method,
        zorder=5
    )

    table_row = []

    for value, error in zip(
        values,
        raw_errors
    ):

        if np.isfinite(value):

            if np.isfinite(error):

                table_row.append(
                    f"{value:.2f} ± {error:.2f}"
                )

            else:

                table_row.append(
                    f"{value:.2f}"
                )

        else:

            table_row.append("N/A")

    table_data.append(table_row)
    row_labels.append(method)
    row_colors.append(METHOD_COLORS[method])


# ============================================================
# EXPERIMENTAL
# ============================================================

experimental_values = np.array([

    EXPERIMENTAL_TAU_PS.get(
        structure,
        np.nan
    )

    for structure in STRUCTURES
])


valid = np.isfinite(
    experimental_values
)


ax.plot(
    x[valid] + EXPERIMENTAL_OFFSET,
    experimental_values[valid],
    "D--",
    color=COLOR_EXPERIMENTAL,
    markersize=MARKER_SIZE,
    linewidth=LINE_WIDTH,
    markeredgewidth=MARKER_EDGE_WIDTH,
    label="Experimental",
    zorder=6
)


for value in experimental_values:

    if np.isfinite(value):
        all_upper_values.append(value)


table_data.append([

    (
        f"{value:.2f}"
        if np.isfinite(value)
        else "N/A"
    )

    for value in experimental_values
])


row_labels.append("Experimental")
row_colors.append(COLOR_EXPERIMENTAL)


# ============================================================
# Y LIMITS
# ============================================================

if len(all_upper_values) == 0:

    raise ValueError(
        "No valid nonradiative lifetime values found."
    )


# ------------------------------------------------------------
# محدوده ثابت محور y
# ------------------------------------------------------------
Y_MAX_FIXED = 600.0   # سقف محور y

y_max = Y_MAX_FIXED


# ------------------------------------------------------------
# y_min بر اساس موقعیت جدول محاسبه می‌شود
# ------------------------------------------------------------
y_min = calculate_table_ymin(
    y_max,
    TABLE_TOP_DATA_Y
)


ax.set_ylim(
    y_min,
    y_max
)

# ============================================================
# AXES
# ============================================================

ax.set_xlabel(
    "Molecule",
    fontsize=FONT_AXIS
)

ax.set_ylabel(
    "Nonradiative lifetime (ps)",
    fontsize=FONT_AXIS,
    labelpad=18
)

ax.set_title(
    "Nonradiative Lifetime Comparison",
    fontsize=FONT_TITLE,
    pad=15
)

ax.set_xticks(x)

ax.set_xticklabels(
    list(STRUCTURES.keys()),
    fontsize=FONT_TICKS,
    rotation=20,
    ha="right"
)

ax.set_xlim(
    -0.45,
    len(STRUCTURES) - 0.55
)

style_axes(ax)


# ============================================================
# Y LABEL
# ============================================================

position_ylabel_above_table(
    ax,
    y_min,
    y_max
)


# ============================================================
# LEGEND
# ============================================================

legend = ax.legend(
    fontsize=FONT_LEGEND,
    ncol=2,
    loc="upper left",
    frameon=True,
    fancybox=True,
    framealpha=0.05,
    borderpad=1.0
)

legend.get_frame().set_edgecolor("black")
legend.get_frame().set_facecolor("white")


# ============================================================
# TABLE
# ============================================================

add_table(
    ax,
    table_data,
    row_labels,
    row_colors,
    list(STRUCTURES.keys())
)


fig.subplots_adjust(
    left=0.13,
    right=0.98,
    top=0.90,
    bottom=0.10
)


# ============================================================
# SAVE
# ============================================================

OUTPUT_FILE = os.path.join(
    OUTPUT_DIR,
    "Nonradiative_lifetime_comparison.png"
)

fig.savefig(
    OUTPUT_FILE,
    dpi=FIG_DPI,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

print(f"Saved: {OUTPUT_FILE}")

## Radiative Quantum Yield (PLQY) Comparison

This cell generates a publication-quality bar chart comparing the calculated Photoluminescence Quantum Yields (PLQYs) for the four adamantane structures across different NAMD methods. Experimental reference values are included for direct comparison.

### Purpose

The primary goal is to visualize how the PLQY, a key metric for optoelectronic applications, is predicted by different theoretical methods (FSSH, FSSH2, IDA, MSDM). This allows for an assessment of the accuracy of each method by comparing the calculated trends with experimental measurements.

### Data Source

This cell **does not perform any new calculations or read raw HDF5 data**. It relies entirely on the pre-processed results stored in the `analysis_results.pkl` file, which was generated by the main analysis cell.

### Methodology

1.  **Data Extraction:**
    - Loads the `analysis_results.pkl` file containing the final fitted parameters for all structure-method combinations.
    - Extracts the non-radiative lifetime ($\tau_{nr}$) from the `final_fit` dictionary.
    - Retrieves the pre-computed radiative lifetime ($\tau_r$) from the `RADIATIVE_TAU_COMPUTED_NS` dictionary stored in the pickle file.

2.  **PLQY Calculation:**
    - Calculates the PLQY using the standard formula:
      $$
      \Phi = \frac{k_r}{k_r + k_{nr}} = \frac{\tau_{nr}}{\tau_{nr} + \tau_r}
      $$
      where $\tau_{nr}$ is the non-radiative lifetime (in ps) and $\tau_r$ is the radiative lifetime (in ns, converted to ps for consistency).
    - The uncertainty in the PLQY is propagated from the bootstrap confidence interval of the non-radiative lifetime.

3.  **Visualization:**
    - Creates a grouped bar chart where:
        - **X-axis:** Represents the four adamantane structures.
        - **Y-axis:** Represents the PLQY in percentage (%).
    - Each method is assigned a distinct color and a slight horizontal offset.
    - Experimental PLQY values are plotted as separate black diamond markers for direct comparison.
    - Error bars represent the propagated 95% confidence interval. The visual size of the errors is capped to prevent large bars from distorting the plot.

4.  **Data Table:**
    - A summary table is automatically generated and placed at the bottom of the figure, providing the exact numerical values (mean ± error) for each calculated PLQY and the experimental reference.

5.  **Output:**
    - The final figure is saved as a high-resolution PNG file (`Radiative_quantum_yield_comparison.png`) in the specified output directory.

In [ ]:
# ============================================================
# CELL 2
# RADIATIVE QUANTUM YIELD / PLQY COMPARISON
#
# Reads only:
#   analysis_results.pkl
#
# No HDF5 reading
# No refitting
# No new analysis
# ============================================================

import os
import pickle
import numpy as np
import matplotlib.pyplot as plt


# ============================================================
# SETTINGS
# ============================================================

OUTPUT_DIR = (
    "/home/hamid/A/NAMD/"
    "analysis_results_batch_average"
)

RESULTS_FILE = os.path.join(
    OUTPUT_DIR,
    "analysis_results.pkl"
)

FIG_DPI = 600


# ============================================================
# STRUCTURES
# ============================================================

STRUCTURES = {
    "Adamantane": "adamantane",
    "1-Methyl-adamantane": "1methyl_adamantane",
    "2-Methyl-adamantane": "2methyl_adamantane",
    "3-Methyl-adamantane": "3methyl_adamantane",
}

METHODS = [
    "MSDM",
    "IDA",
    "FSSH",
    "FSSH2",
    
    
]


# ============================================================
# COLORS
# ============================================================

METHOD_COLORS = {
    "FSSH": "blue",
    "FSSH2": "red",
    "IDA": "green",
    "MSDM": "purple",
}

COLOR_EXPERIMENTAL = "black"


# ============================================================
# FIGURE STYLE
# ============================================================

FIGSIZE = (16, 12)

FONT_TITLE = 28
FONT_AXIS = 24
FONT_TICKS = 20
FONT_LEGEND = 19

FONT_TABLE_HEADER = 18
FONT_TABLE_TEXT = 16
FONT_TABLE_ROW_LABEL = 16

MARKER_SIZE = 14

LINE_WIDTH = 0.5

ERROR_LINE_WIDTH = 2.0
ERROR_CAPSIZE = 7
ERROR_CAPTHICK = 1.0
MARKER_EDGE_WIDTH = 1.0


# ============================================================
# HORIZONTAL OFFSETS
# ============================================================

METHOD_OFFSETS = np.linspace(
    -0.09,
    0.09,
    len(METHODS)
)

EXPERIMENTAL_OFFSET = 0.13


# ============================================================
# TABLE
# ============================================================

TABLE_BBOX_BOTTOM = 0.0

TABLE_BBOX_HEIGHT = 0.38

# Independent position for PLQY table
TABLE_TOP_DATA_Y = -2.0


# ============================================================
# ERROR LIMIT
# ============================================================

PLQY_ERROR_FRACTION_LIMIT = 0.75


# ============================================================
# LOAD SAVED DATA
# ============================================================

if not os.path.isfile(RESULTS_FILE):

    raise FileNotFoundError(
        f"Could not find:\n{RESULTS_FILE}"
    )


with open(
    RESULTS_FILE,
    "rb"
) as f:

    saved_data = pickle.load(f)


# ============================================================
# EXTRACT all_results
# ============================================================

if (
    isinstance(saved_data, dict)
    and
    "all_results" in saved_data
):

    all_results = saved_data["all_results"]

else:

    all_results = saved_data


# ============================================================
# EXPERIMENTAL PLQY
# ============================================================

EXPERIMENTAL_PLQY_PERCENT = (

    saved_data.get(
        "experimental_plqy_percent",
        {}
    )

    if isinstance(saved_data, dict)

    else {}
)


# ============================================================
# RADIATIVE LIFETIME
# ============================================================

RADIATIVE_TAU_COMPUTED_NS = (

    saved_data.get(
        "radiative_tau_computed_ns",
        {}
    )

    if isinstance(saved_data, dict)

    else {}
)


# ============================================================
# HELPER: ACCESS ONE RESULT
# ============================================================

def get_result(
    structure,
    method
):

    try:

        return all_results[
            (
                structure,
                method
            )
        ]

    except Exception:

        pass

    try:

        return all_results[
            structure
        ][
            method
        ]

    except Exception:

        return None


# ============================================================
# HELPER: LIMIT GRAPHICAL ERROR
# ============================================================

def limit_error(
    value,
    error,
    fraction_limit=None,
    absolute_limit=None
):

    if (
        not np.isfinite(value)
        or
        not np.isfinite(error)
    ):

        return np.nan

    limited = abs(error)

    if fraction_limit is not None:

        limited = min(
            limited,
            abs(value) * fraction_limit
        )

    if absolute_limit is not None:

        limited = min(
            limited,
            absolute_limit
        )

    return max(
        0.0,
        limited
    )


# ============================================================
# TABLE Y-MIN
# ============================================================

def calculate_table_ymin(
    y_max,
    table_top_data_y
):

    return (
        table_top_data_y
        -
        TABLE_BBOX_HEIGHT * y_max
    ) / (
        1.0 -
        TABLE_BBOX_HEIGHT
    )


# ============================================================
# POSITION Y LABEL
# ============================================================

def position_ylabel_above_table(
    ax,
    y_min,
    y_max
):

    table_top = (
        y_min * (1.0 - TABLE_BBOX_HEIGHT)
        +
        y_max * TABLE_BBOX_HEIGHT
    )

    table_top_axes = (
        table_top - y_min
    ) / (
        y_max - y_min
    )

    y_center_axes = (
        table_top_axes + 1.0
    ) / 2.0

    ax.yaxis.set_label_coords(
        -0.08,
        y_center_axes
    )


# ============================================================
# REMOVE NEGATIVE Y TICKS
# ============================================================

def remove_negative_yticks(
    ax
):

    ticks = ax.get_yticks()

    positive_ticks = [
        tick
        for tick in ticks
        if tick >= 0
    ]

    ax.set_yticks(
        positive_ticks
    )


# ============================================================
# COMMON AXIS STYLE
# ============================================================

def style_axes(
    ax
):

    ax.tick_params(
        axis="both",
        which="major",
        labelsize=FONT_TICKS,
        width=1.2,
        length=6
    )

    ax.tick_params(
        axis="y",
        pad=8
    )

    remove_negative_yticks(
        ax
    )


# ============================================================
# ADD TABLE
# ============================================================

def add_table(
    ax,
    table_data,
    row_labels,
    row_colors,
    col_labels
):

    table = ax.table(

        cellText=table_data,

        colLabels=col_labels,

        rowLabels=row_labels,

        loc="center",

        cellLoc="center",

        rowLoc="center",

        colColours=[
            "#eaeaea"
            for _ in col_labels
        ],

        bbox=[
            TABLE_BBOX_BOTTOM,
            0.0,
            1.0,
            TABLE_BBOX_HEIGHT
        ]
    )


    table.auto_set_font_size(
        False
    )


    table.set_fontsize(
        FONT_TABLE_TEXT
    )


    for (
        row,
        col
    ), cell in table.get_celld().items():


        # ====================================================
        # HEADER
        # ====================================================

        if row == 0:

            cell.get_text().set_fontsize(
                FONT_TABLE_HEADER
            )

            cell.get_text().set_fontweight(
                "normal"
            )

            cell.get_text().set_color(
                "black"
            )

            cell.set_facecolor(
                "#eaeaea"
            )


        # ====================================================
        # DATA ROWS
        # ====================================================

        else:

            cell.get_text().set_fontsize(
                FONT_TABLE_TEXT
            )


            # =================================================
            # METHOD NAME COLUMN
            # =================================================

            if col == -1:

                cell.set_facecolor(
                    "#eaeaea"
                )

                cell.get_text().set_fontweight(
                    "normal"
                )

                if (
                    row - 1
                    <
                    len(row_colors)
                ):

                    cell.get_text().set_color(
                        row_colors[row - 1]
                    )

                else:

                    cell.get_text().set_color(
                        "black"
                    )

                cell.get_text().set_fontsize(
                    FONT_TABLE_ROW_LABEL
                )


            # =================================================
            # NUMERIC CELLS
            # =================================================

            else:

                if (
                    row - 1
                    <
                    len(row_colors)
                ):

                    cell.get_text().set_color(
                        row_colors[row - 1]
                    )


        cell.set_edgecolor(
            "black"
        )

        cell.set_linewidth(
            1.0
        )


    return table


# ============================================================
# FIGURE
# ============================================================

fig, ax = plt.subplots(
    figsize=FIGSIZE
)


x = np.arange(
    len(STRUCTURES)
)


table_data = []

row_labels = []

row_colors = []

all_upper_values = []


# ============================================================
# METHODS
# ============================================================

for i, method in enumerate(METHODS):

    values = []

    raw_errors = []

    visual_errors = []


    for structure in STRUCTURES:

        result = get_result(
            structure,
            method
        )


        if result is None:

            values.append(np.nan)
            raw_errors.append(np.nan)
            visual_errors.append(np.nan)

            continue


        final_fit = result.get(
            "final_fit"
        )

        plqy = result.get(
            "plqy"
        )


        # ====================================================
        # PLQY VALUE
        # ====================================================

        if (
            final_fit is not None
            and
            structure in RADIATIVE_TAU_COMPUTED_NS
        ):

            tau_nr_ps = float(
                final_fit.get(
                    "tau_ps",
                    np.nan
                )
            )


            tau_r_ns = float(
                RADIATIVE_TAU_COMPUTED_NS[
                    structure
                ]
            )


            if (
                np.isfinite(tau_nr_ps)
                and
                np.isfinite(tau_r_ns)
            ):

                tau_r_ps = (
                    tau_r_ns * 1000.0
                )


                value = (
                    100.0
                    *
                    tau_nr_ps
                    /
                    (
                        tau_nr_ps
                        +
                        tau_r_ps
                    )
                )

            else:

                value = np.nan

        else:

            value = np.nan


        # ====================================================
        # PLQY CI
        # ====================================================

        if (
            plqy is not None
            and
            "ci" in plqy
        ):

            ci = plqy["ci"]


            try:

                ci_low = float(
                    ci[0]
                )

                ci_high = float(
                    ci[1]
                )


                error = max(
                    abs(value - ci_low),
                    abs(ci_high - value)
                )


            except Exception:

                error = np.nan


        else:

            error = np.nan


        visual_error = limit_error(

            value,

            error,

            fraction_limit=(
                PLQY_ERROR_FRACTION_LIMIT
            )
        )


        values.append(
            value
        )

        raw_errors.append(
            error
        )

        visual_errors.append(
            visual_error
        )


        if (
            np.isfinite(value)
            and
            np.isfinite(visual_error)
        ):

            all_upper_values.append(
                value + visual_error
            )


    values = np.asarray(
        values
    )

    raw_errors = np.asarray(
        raw_errors
    )

    visual_errors = np.asarray(
        visual_errors
    )


    # ========================================================
    # PLOT
    # ========================================================

    ax.errorbar(

        x + METHOD_OFFSETS[i],

        values,

        yerr=visual_errors,

        fmt="s-",

        color=METHOD_COLORS[method],

        markersize=MARKER_SIZE,

        linewidth=LINE_WIDTH,

        elinewidth=ERROR_LINE_WIDTH,

        capsize=ERROR_CAPSIZE,

        capthick=ERROR_CAPTHICK,

        markeredgewidth=MARKER_EDGE_WIDTH,

        label=method,

        zorder=5
    )


    # ========================================================
    # TABLE ROW
    # ========================================================

    table_row = []


    for value, error in zip(
        values,
        raw_errors
    ):

        if np.isfinite(value):

            if np.isfinite(error):

                table_row.append(
                    f"{value:.2f} ± {error:.2f}"
                )

            else:

                table_row.append(
                    f"{value:.2f}"
                )

        else:

            table_row.append(
                "N/A"
            )


    table_data.append(
        table_row
    )

    row_labels.append(
        method
    )

    row_colors.append(
        METHOD_COLORS[method]
    )


# ============================================================
# EXPERIMENTAL PLQY
# ============================================================

experimental_plqy = np.array([

    EXPERIMENTAL_PLQY_PERCENT.get(
        structure,
        np.nan
    )

    for structure in STRUCTURES
])


valid = np.isfinite(
    experimental_plqy
)


if np.any(valid):

    ax.plot(

        x[valid] + EXPERIMENTAL_OFFSET,

        experimental_plqy[valid],

        "D--",

        color=COLOR_EXPERIMENTAL,

        markersize=MARKER_SIZE,

        linewidth=LINE_WIDTH,

        markeredgewidth=MARKER_EDGE_WIDTH,

        label="Experimental",

        zorder=6
    )


    for value in experimental_plqy:

        if np.isfinite(value):

            all_upper_values.append(
                value
            )


    table_data.append([

        (
            f"{value:.2f}"
            if np.isfinite(value)
            else "N/A"
        )

        for value in experimental_plqy
    ])


    row_labels.append(
        "Experimental"
    )

    row_colors.append(
        COLOR_EXPERIMENTAL
    )


# ============================================================
# Y LIMITS
# ============================================================

if not all_upper_values:

    raise ValueError(
        "No valid PLQY values were found."
    )


y_max = max(
    all_upper_values
) * 1.12


if y_max <= 0:

    y_max = 1.0


y_min = calculate_table_ymin(

    y_max,

    TABLE_TOP_DATA_Y
)


ax.set_ylim(
    y_min,
    y_max
)


# ============================================================
# AXES
# ============================================================

ax.set_xlabel(
    "Molecule",
    fontsize=FONT_AXIS
)

ax.set_ylabel(
    "Radiative quantum yield (%)",
    fontsize=FONT_AXIS,
    labelpad=18
)

ax.set_title(
    "Radiative Quantum Yield Comparison",
    fontsize=FONT_TITLE,
    pad=15
)

ax.set_xticks(x)

ax.set_xticklabels(
    list(STRUCTURES.keys()),
    fontsize=FONT_TICKS,
    rotation=20,
    ha="right"
)

ax.set_xlim(
    -0.45,
    len(STRUCTURES) - 0.55
)


style_axes(ax)


# ============================================================
# Y LABEL
# ============================================================

position_ylabel_above_table(
    ax,
    y_min,
    y_max
)


# ============================================================
# LEGEND
# ============================================================

legend = ax.legend(

    fontsize=FONT_LEGEND,

    ncol=2,

    loc="upper left",

    frameon=True,

    fancybox=True,

    framealpha=0.1,

    borderpad=1.0
)

legend.get_frame().set_edgecolor(
    "black"
)

legend.get_frame().set_facecolor(
    "white"
)


# ============================================================
# TABLE
# ============================================================

add_table(

    ax,

    table_data,

    row_labels,

    row_colors,

    list(STRUCTURES.keys())
)


fig.subplots_adjust(

    left=0.13,

    right=0.98,

    top=0.90,

    bottom=0.10
)


# ============================================================
# SAVE
# ============================================================

OUTPUT_FILE = os.path.join(

    OUTPUT_DIR,

    "Radiative_quantum_yield_comparison.png"
)


fig.savefig(

    OUTPUT_FILE,

    dpi=FIG_DPI,

    bbox_inches="tight"
)


plt.show()

plt.close(fig)


print(
    f"Saved: {OUTPUT_FILE}"
)

## β (Stretch Exponent) Comparison

This cell generates a publication-quality bar chart comparing the stretch exponent ($\beta$) obtained from fitting the ground-state population recovery dynamics to a stretched exponential model for all four adamantane structures and NAMD methods.

### Purpose

The stretch exponent $\beta$ quantifies the degree of non-exponentiality (or dispersity) in the non-radiative relaxation kinetics. A value of $\beta = 1$ corresponds to a simple single-exponential decay, while $\beta < 1$ indicates a distribution of relaxation timescales, which is common in disordered or complex molecular systems. This plot allows for a systematic comparison of how different theoretical methods predict this dispersive behavior.

### Data Source

This cell **does not perform any new calculations or read raw data**. It relies entirely on the pre-processed results stored in the `analysis_results.pkl` file, which was generated by the main analysis cell.

### Methodology

1.  **Data Extraction:**
    - Loads the `analysis_results.pkl` file containing the final fitted parameters for all structure-method combinations.
    - Extracts the stretch exponent ($\beta$) and its confidence interval (CI) from the `final_fit` and `bootstrap` dictionaries for each case.

2.  **Visualization:**
    - Creates a grouped bar chart where:
        - **X-axis:** Represents the four adamantane structures.
        - **Y-axis:** Represents the value of the stretch exponent $\beta$.
    - Each method is assigned a distinct color and a slight horizontal offset.
    - Error bars represent the 95% confidence interval derived from the bootstrap analysis. The visual size of these errors is capped by both a relative fraction (`BETA_ERROR_FRACTION_LIMIT`) and an absolute value (`BETA_ABSOLUTE_ERROR_LIMIT`) to prevent excessively large bars from distorting the plot.

3.  **Data Table:**
    - A summary table is automatically generated and placed at the bottom of the figure, providing the exact numerical values (mean ± error) for each calculated $\beta$.

4.  **Output:**
    - The final figure is saved as a high-resolution PNG file (`Beta_comparison.png`) in the specified output directory.

In [ ]:
# ============================================================
# CELL 3
# BETA COMPARISON
#
# Reads only:
#   analysis_results.pkl
#
# No HDF5 reading
# No refitting
# No new analysis
# ============================================================

import os
import pickle
import numpy as np
import matplotlib.pyplot as plt


# ============================================================
# SETTINGS
# ============================================================

OUTPUT_DIR = (
    "/home/hamid/A/NAMD/"
    "analysis_results_batch_average"
)

RESULTS_FILE = os.path.join(
    OUTPUT_DIR,
    "analysis_results.pkl"
)

FIG_DPI = 600


# ============================================================
# STRUCTURES
# ============================================================

STRUCTURES = {
    "Adamantane": "adamantane",
    "1-Methyl-adamantane": "1methyl_adamantane",
    "2-Methyl-adamantane": "2methyl_adamantane",
    "3-Methyl-adamantane": "3methyl_adamantane",
}

METHODS = [
    "FSSH2",
    "FSSH",
    "MSDM",
    "IDA",
]


# ============================================================
# COLORS
# ============================================================

METHOD_COLORS = {
    "FSSH": "blue",
    "FSSH2": "red",
    "IDA": "green",
    "MSDM": "purple",
}

COLOR_EXPERIMENTAL = "black"


# ============================================================
# FIGURE STYLE
# ============================================================

FIGSIZE = (16, 12)

FONT_TITLE = 28
FONT_AXIS = 24
FONT_TICKS = 20
FONT_LEGEND = 19

FONT_TABLE_HEADER = 18
FONT_TABLE_TEXT = 16
FONT_TABLE_ROW_LABEL = 16

MARKER_SIZE = 14

LINE_WIDTH = 0.5

ERROR_LINE_WIDTH = 2.0
ERROR_CAPSIZE = 7
ERROR_CAPTHICK = 1.0
MARKER_EDGE_WIDTH = 1.0


# ============================================================
# HORIZONTAL OFFSETS
# ============================================================

METHOD_OFFSETS = np.linspace(
    -0.09,
    0.09,
    len(METHODS)
)


# ============================================================
# TABLE
# ============================================================

TABLE_BBOX_BOTTOM = 0.0

TABLE_BBOX_HEIGHT = 0.38

# Independent position for Beta table
TABLE_TOP_DATA_Y = 0.0


# ============================================================
# ERROR LIMITS
# ============================================================

BETA_ERROR_FRACTION_LIMIT = 0.75
BETA_ABSOLUTE_ERROR_LIMIT = 1.0


# ============================================================
# LOAD SAVED DATA
# ============================================================

if not os.path.isfile(RESULTS_FILE):

    raise FileNotFoundError(
        f"Could not find:\n{RESULTS_FILE}"
    )


with open(
    RESULTS_FILE,
    "rb"
) as f:

    saved_data = pickle.load(f)


# ============================================================
# EXTRACT all_results
# ============================================================

if (
    isinstance(saved_data, dict)
    and
    "all_results" in saved_data
):

    all_results = saved_data["all_results"]

else:

    all_results = saved_data


# ============================================================
# HELPER: ACCESS ONE RESULT
# ============================================================

def get_result(
    structure,
    method
):

    try:

        return all_results[
            (
                structure,
                method
            )
        ]

    except Exception:

        pass

    try:

        return all_results[
            structure
        ][
            method
        ]

    except Exception:

        return None


# ============================================================
# HELPER: SYMMETRIC ERROR FROM CI
# ============================================================

def symmetric_error_from_ci(
    ci
):

    if ci is None:

        return np.nan


    try:

        low = float(
            ci[0]
        )

        high = float(
            ci[1]
        )

    except Exception:

        return np.nan


    if (
        not np.isfinite(low)
        or
        not np.isfinite(high)
    ):

        return np.nan


    return abs(
        high - low
    ) / 2.0


# ============================================================
# HELPER: LIMIT GRAPHICAL ERROR
# ============================================================

def limit_error(
    value,
    error,
    fraction_limit=None,
    absolute_limit=None
):

    if (
        not np.isfinite(value)
        or
        not np.isfinite(error)
    ):

        return np.nan


    limited = abs(error)


    if fraction_limit is not None:

        limited = min(

            limited,

            abs(value)
            *
            fraction_limit
        )


    if absolute_limit is not None:

        limited = min(

            limited,

            absolute_limit
        )


    return max(
        0.0,
        limited
    )


# ============================================================
# TABLE Y-MIN
# ============================================================

def calculate_table_ymin(
    y_max,
    table_top_data_y
):

    return (

        table_top_data_y

        -

        TABLE_BBOX_HEIGHT
        *
        y_max

    ) / (

        1.0
        -
        TABLE_BBOX_HEIGHT

    )


# ============================================================
# POSITION Y LABEL
# ============================================================

def position_ylabel_above_table(
    ax,
    y_min,
    y_max
):

    table_top = (

        y_min
        *
        (
            1.0
            -
            TABLE_BBOX_HEIGHT
        )

        +

        y_max
        *
        TABLE_BBOX_HEIGHT

    )


    table_top_axes = (

        table_top
        -
        y_min

    ) / (

        y_max
        -
        y_min

    )


    y_center_axes = (

        table_top_axes
        +
        1.0

    ) / 2.0


    ax.yaxis.set_label_coords(

        -0.08,

        y_center_axes

    )


# ============================================================
# REMOVE NEGATIVE Y TICKS
# ============================================================

def remove_negative_yticks(
    ax
):

    ticks = ax.get_yticks()


    positive_ticks = [

        tick

        for tick in ticks

        if tick >= 0

    ]


    ax.set_yticks(
        positive_ticks
    )


# ============================================================
# COMMON AXIS STYLE
# ============================================================

def style_axes(
    ax
):

    ax.tick_params(

        axis="both",

        which="major",

        labelsize=FONT_TICKS,

        width=1.2,

        length=6

    )


    ax.tick_params(

        axis="y",

        pad=8

    )


    remove_negative_yticks(
        ax
    )


# ============================================================
# ADD TABLE
# ============================================================

def add_table(
    ax,
    table_data,
    row_labels,
    row_colors,
    col_labels
):

    table = ax.table(

        cellText=table_data,

        colLabels=col_labels,

        rowLabels=row_labels,

        loc="center",

        cellLoc="center",

        rowLoc="center",

        colColours=[

            "#eaeaea"

            for _ in col_labels

        ],

        bbox=[

            TABLE_BBOX_BOTTOM,

            0.0,

            1.0,

            TABLE_BBOX_HEIGHT

        ]

    )


    table.auto_set_font_size(
        False
    )


    table.set_fontsize(
        FONT_TABLE_TEXT
    )


    for (
        row,
        col
    ), cell in table.get_celld().items():


        # ====================================================
        # HEADER
        # ====================================================

        if row == 0:

            cell.get_text().set_fontsize(

                FONT_TABLE_HEADER

            )


            cell.get_text().set_fontweight(

                "normal"

            )


            cell.get_text().set_color(

                "black"

            )


            cell.set_facecolor(

                "#eaeaea"

            )


        # ====================================================
        # DATA ROWS
        # ====================================================

        else:

            cell.get_text().set_fontsize(

                FONT_TABLE_TEXT

            )


            # =================================================
            # METHOD NAME COLUMN
            # =================================================

            if col == -1:

                cell.set_facecolor(

                    "#eaeaea"

                )


                cell.get_text().set_fontweight(

                    "normal"

                )


                if (

                    row - 1

                    <

                    len(row_colors)

                ):

                    cell.get_text().set_color(

                        row_colors[row - 1]

                    )

                else:

                    cell.get_text().set_color(

                        "black"

                    )


                cell.get_text().set_fontsize(

                    FONT_TABLE_ROW_LABEL

                )


            # =================================================
            # NUMERIC CELLS
            # =================================================

            else:

                if (

                    row - 1

                    <

                    len(row_colors)

                ):

                    cell.get_text().set_color(

                        row_colors[row - 1]

                    )


        cell.set_edgecolor(
            "black"
        )

        cell.set_linewidth(
            1.0
        )


    return table


# ============================================================
# FIGURE
# ============================================================

fig, ax = plt.subplots(

    figsize=FIGSIZE

)


x = np.arange(

    len(STRUCTURES)

)


table_data = []

row_labels = []

row_colors = []

all_upper_values = []


# ============================================================
# METHODS
# ============================================================

for i, method in enumerate(METHODS):

    values = []

    raw_errors = []

    visual_errors = []


    for structure in STRUCTURES:

        result = get_result(

            structure,

            method

        )


        if result is None:

            values.append(np.nan)

            raw_errors.append(np.nan)

            visual_errors.append(np.nan)

            continue


        final_fit = result.get(

            "final_fit"

        )


        bootstrap = result.get(

            "bootstrap"

        )


        # ====================================================
        # BETA VALUE
        # ====================================================

        if final_fit is not None:

            value = float(

                final_fit.get(

                    "beta",

                    np.nan

                )

            )

        else:

            value = np.nan


        # ====================================================
        # BETA CI
        # ====================================================

        if bootstrap is not None:

            ci = bootstrap.get(

                "beta_ci"

            )

        else:

            ci = None


        error = symmetric_error_from_ci(

            ci

        )


        visual_error = limit_error(

            value,

            error,

            fraction_limit=(

                BETA_ERROR_FRACTION_LIMIT

            ),

            absolute_limit=(

                BETA_ABSOLUTE_ERROR_LIMIT

            )

        )


        values.append(value)

        raw_errors.append(error)

        visual_errors.append(

            visual_error

        )


        if (

            np.isfinite(value)

            and

            np.isfinite(visual_error)

        ):

            all_upper_values.append(

                value + visual_error

            )


    values = np.asarray(

        values

    )


    raw_errors = np.asarray(

        raw_errors

    )


    visual_errors = np.asarray(

        visual_errors

    )


    # ========================================================
    # PLOT
    # ========================================================

    ax.errorbar(

        x + METHOD_OFFSETS[i],

        values,

        yerr=visual_errors,

        fmt="s-",

        color=METHOD_COLORS[method],

        markersize=MARKER_SIZE,

        linewidth=LINE_WIDTH,

        elinewidth=ERROR_LINE_WIDTH,

        capsize=ERROR_CAPSIZE,

        capthick=ERROR_CAPTHICK,

        markeredgewidth=MARKER_EDGE_WIDTH,

        label=method,

        zorder=5

    )


    # ========================================================
    # TABLE ROW
    # ========================================================

    table_row = []


    for value, error in zip(

        values,

        raw_errors

    ):

        if np.isfinite(value):

            if np.isfinite(error):

                table_row.append(

                    f"{value:.3f} ± {error:.3f}"

                )

            else:

                table_row.append(

                    f"{value:.3f}"

                )

        else:

            table_row.append(

                "N/A"

            )


    table_data.append(

        table_row

    )


    row_labels.append(

        method

    )


    row_colors.append(

        METHOD_COLORS[method]

    )



# ============================================================
# Y LIMITS
# ============================================================

if len(all_upper_values) == 0:

    raise ValueError(

        "No valid beta values found."

    )


y_max = max(

    all_upper_values

) * 1.15


if y_max <= 0:

    y_max = 1.0


y_min = calculate_table_ymin(

    y_max,

    TABLE_TOP_DATA_Y

)


ax.set_ylim(

    y_min,

    y_max

)


# ============================================================
# AXES
# ============================================================

ax.set_xlabel(

    "Molecule",

    fontsize=FONT_AXIS

)


ax.set_ylabel(

    r"$\beta$",

    fontsize=FONT_AXIS,

    labelpad=18

)


ax.set_title(

    r"$\beta$ Comparison",

    fontsize=FONT_TITLE,

    pad=15

)


ax.set_xticks(x)


ax.set_xticklabels(

    list(STRUCTURES.keys()),

    fontsize=FONT_TICKS,

    rotation=20,

    ha="right"

)


ax.set_xlim(

    -0.45,

    len(STRUCTURES) - 0.55

)


style_axes(ax)


# ============================================================
# Y LABEL
# ============================================================

position_ylabel_above_table(

    ax,

    y_min,

    y_max

)


# ============================================================
# LEGEND
# ============================================================

legend = ax.legend(

    fontsize=FONT_LEGEND,

    ncol=2,

    loc="upper left",

    frameon=True,

    fancybox=True,

    framealpha=0.1,

    borderpad=1.0

)


legend.get_frame().set_edgecolor(

    "black"

)


legend.get_frame().set_facecolor(

    "white"

)


# ============================================================
# TABLE
# ============================================================

add_table(

    ax,

    table_data,

    row_labels,

    row_colors,

    list(STRUCTURES.keys())

)


fig.subplots_adjust(

    left=0.13,

    right=0.98,

    top=0.90,

    bottom=0.10

)


# ============================================================
# SAVE
# ============================================================

OUTPUT_FILE = os.path.join(

    OUTPUT_DIR,

    "Beta_comparison.png"

)


fig.savefig(

    OUTPUT_FILE,

    dpi=FIG_DPI,

    bbox_inches="tight"

)


plt.show()

plt.close(fig)


print(

    f"Saved: {OUTPUT_FILE}"

)

## 4×4 Grid Plot: Methods vs. Structures

This cell generates the main publication-quality figure of the analysis: a 4×4 grid of panels showing the ground-state (S₀) population recovery dynamics for every combination of NAMD method and adamantane structure.

### Purpose

The primary goal is to provide a comprehensive, at-a-glance comparison of how the non-radiative relaxation dynamics depend on both the **theoretical method** (FSSH, FSSH2, IDA, MSDM) and the **molecular structure** (Adamantane, 1-Methyl-, 2-Methyl-, 3-Methyl-adamantane). This figure is designed to be the central visual result of the project.

### Data Source

This cell **does not perform any new calculations or read raw HDF5 data**. It relies entirely on the pre-processed results stored in the `analysis_results.pkl` file.

### Figure Layout

- **Rows:** Each row corresponds to one NAMD method.
- **Columns:** Each column corresponds to one adamantane structure.
- **Panels:** Each of the 16 panels shows the following information for one structure-method combination:
    1.  **Gray Lines:** Individual trajectory batches that did **not** meet the $R^2$ threshold (rejected).
    2.  **Blue Lines:** Individual trajectory batches that **did** meet the $R^2$ threshold (accepted).
    3.  **Red Solid Line:** The final ensemble-averaged fit to the stretched exponential model.
    4.  **Red Shaded Region & Dashed Lines:** The 95% confidence interval from the bootstrap analysis.
    5.  **Legend:** Shows the fitted non-radiative lifetime ($\tau$) with its bootstrap error and the minimum $R^2$ threshold used for batch selection.

### Key Features

- **Independent Y-Axis Scaling:** Each panel has its own Y-axis scale (`sharey=False`). This is crucial because the magnitude of the S₀ population varies significantly between different structures and methods (especially for FSSH/IDA vs. MSDM), and a shared scale would make it impossible to see the details of the faster or smaller-amplitude dynamics.
- **R² Criterion in Legend:** The legend explicitly states the $R^2$ threshold that was applied to filter the batches, providing transparency about the data quality.
- **Consistent X-Axis:** All panels share the same X-axis (time in fs, 0–3000 fs) for direct temporal comparison.
- **Common Labels:** A single, large X-axis label ("Time (fs)") and Y-axis label ("Population (S₀)") are applied to the entire figure, along with a common title.
- **Publication-Quality Styling:** Uses thick axis lines, large fonts, and clean legends suitable for journal submission.

### Output

The final figure is saved as a high-resolution PNG file (`figure_4x4_methods_vs_structures.png`) in the specified output directory, ready for inclusion in a manuscript or presentation.

In [ ]:
#!/usr/bin/env python3
# ============================================================
# PLOT 4x4 GRID FROM SAVED ANALYSIS RESULTS
#
# Rows    = methods
# Columns = structures
#
# Publication-quality version
#
# Updated:
#   - Independent Y-axis scale for every panel
#   - R² criterion shown in legend
#   - Legend stays inside each panel
#   - Smaller Y-axis tick labels
#   - Larger / tighter panels
#   - Common title closer to the figure
#   - Common X/Y labels closer to the panels
#   - Method name REMOVED from legend
#   - τ and its error are now LARGER in the legend
#   - Main title: larger, NOT bold, centered with equal margins
#   - Axis labels: NOT bold, larger, closer to panels
# ============================================================

import os
import pickle
import numpy as np
import matplotlib.pyplot as plt

from matplotlib.ticker import MaxNLocator, FormatStrFormatter


# ============================================================
# PUBLICATION-QUALITY MATPLOTLIB SETTINGS
# ============================================================

plt.rcParams.update({
    "font.family": "DejaVu Sans",

    # Main text
    "font.size": 20,

    # Axes
    "axes.labelsize": 24,
    "axes.titlesize": 24,

    # Tick labels
    "xtick.labelsize": 22,
    "ytick.labelsize": 20,

    # Legend
    "legend.fontsize": 15,

    # Axis thickness
    "axes.linewidth": 2.5,

    # White background
    "figure.facecolor": "white",
    "axes.facecolor": "white",

    # Math font
    "mathtext.fontset": "dejavusans",

    # Saving
    "savefig.dpi": 600,
})


# ============================================================
# PATHS
# ============================================================

BASE_DIR = "/home/hamid/A/NAMD"

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "analysis_results_batch_average"
)

RESULTS_PKL = os.path.join(
    OUTPUT_DIR,
    "analysis_results.pkl"
)

FIGURE_OUT = os.path.join(
    OUTPUT_DIR,
    "figure_4x4_methods_vs_structures.png"
)


# ============================================================
# COMMON FIGURE TITLE
# ============================================================

FIGURE_TITLE = (
    "Nonradiative S$_0$  Recovery: Structures vs. Methods "
    
)


# ============================================================
# FITTING FUNCTION
# ============================================================

def stretched_exp(t, tau, beta):

    t = np.asarray(
        t,
        dtype=float
    )

    tau = max(
        float(tau),
        1e-12
    )

    return (
        1.0
        - np.exp(
            -np.power(
                np.maximum(
                    t,
                    0.0
                ) / tau,
                float(beta)
            )
        )
    )


# ============================================================
# LOAD SAVED RESULTS
# ============================================================

with open(
    RESULTS_PKL,
    "rb"
) as f:

    saved_data = pickle.load(f)


all_results = saved_data[
    "all_results"
]

STRUCTURES = saved_data[
    "structures"
]

METHODS = saved_data[
    "methods"
]

R2_THRESHOLDS = saved_data[
    "r2_thresholds"
]

CONF_LEVEL = saved_data[
    "confidence_level"
]

MAX_TIME_FS = saved_data[
    "max_time_fs"
]

EXP_TAU_PS = saved_data[
    "experimental_tau_ps"
]


structure_names = list(
    STRUCTURES.keys()
)

method_names = list(
    METHODS
)


# ============================================================
# FIGURE
#
# IMPORTANT:
# sharey=False because each panel has its own Y scale.
# ============================================================

n_rows = len(
    method_names
)

n_cols = len(
    structure_names
)


fig, axes = plt.subplots(
    n_rows,
    n_cols,

    figsize=(
        6.2 * n_cols,
        5.0 * n_rows
    ),

    sharex=True,
    sharey=False,

    squeeze=False,
)


# ============================================================
# HELPER:
# TAU + BOOTSTRAP ERROR
# ============================================================

def get_tau_ps_and_error(result):

    fit = result.get(
        "final_fit"
    )

    boot = result.get(
        "bootstrap"
    )

    if fit is None:

        return (
            np.nan,
            np.nan
        )


    tau_ps = fit[
        "tau_ps"
    ]


    if boot is not None:

        err_ps = (
            0.5
            * (
                boot["tau_ci"][1]
                -
                boot["tau_ci"][0]
            )
        )

    else:

        err_ps = np.nan


    return (
        tau_ps,
        err_ps
    )


# ============================================================
# HELPER:
# DETERMINE INDEPENDENT Y LIMIT
# ============================================================

def determine_panel_ylim(
    result,
    padding=0.08
):

    values = []


    # --------------------------------------------------------
    # Accepted trajectories
    # --------------------------------------------------------

    for batch in result.get(
        "accepted_batches",
        []
    ):

        y = np.asarray(
            batch["population"],
            dtype=float
        )

        y = y[
            np.isfinite(y)
        ]

        if len(y) > 0:

            values.append(
                np.max(y)
            )


    # --------------------------------------------------------
    # Rejected trajectories
    # --------------------------------------------------------

    for batch in result.get(
        "rejected_batches",
        []
    ):

        y = np.asarray(
            batch["population"],
            dtype=float
        )

        y = y[
            np.isfinite(y)
        ]

        if len(y) > 0:

            values.append(
                np.max(y)
            )


    # --------------------------------------------------------
    # Ensemble
    # --------------------------------------------------------

    ensemble = result.get(
        "ensemble"
    )

    if ensemble is not None:

        y = np.asarray(
            ensemble.get(
                "population",
                []
            ),
            dtype=float
        )

        y = y[
            np.isfinite(y)
        ]

        if len(y) > 0:

            values.append(
                np.max(y)
            )


    # --------------------------------------------------------
    # Fitted curve
    # --------------------------------------------------------

    fit = result.get(
        "final_fit"
    )

    if (
        fit is not None
        and ensemble is not None
    ):

        t_grid = np.asarray(
            ensemble["time_fs"],
            dtype=float
        )

        fit_y = stretched_exp(
            t_grid,
            fit["tau_fs"],
            fit["beta"]
        )

        fit_y = fit_y[
            np.isfinite(fit_y)
        ]

        if len(fit_y) > 0:

            values.append(
                np.max(fit_y)
            )


    # --------------------------------------------------------
    # Bootstrap
    # --------------------------------------------------------

    boot = result.get(
        "bootstrap"
    )

    if boot is not None:

        for key in (
            "fit_lower",
            "fit_upper",
        ):

            if key in boot:

                y = np.asarray(
                    boot[key],
                    dtype=float
                )

                y = y[
                    np.isfinite(y)
                ]

                if len(y) > 0:

                    values.append(
                        np.max(y)
                    )


    # --------------------------------------------------------
    # Fallback
    # --------------------------------------------------------

    if len(values) == 0:

        return (
            0.0,
            1.0
        )


    ymax = max(
        values
    )


    # --------------------------------------------------------
    # Add small margin
    # --------------------------------------------------------

    ymax *= (
        1.0 + padding
    )


    # --------------------------------------------------------
    # Round upward to a clean value
    # --------------------------------------------------------

    exponent = np.floor(
        np.log10(ymax)
    )

    scale = (
        10 ** exponent
    )

    normalized = (
        ymax / scale
    )


    if normalized <= 1.0:

        nice = 1.0

    elif normalized <= 1.5:

        nice = 1.5

    elif normalized <= 2.0:

        nice = 2.0

    elif normalized <= 2.5:

        nice = 2.5

    elif normalized <= 3.0:

        nice = 3.0

    elif normalized <= 4.0:

        nice = 4.0

    elif normalized <= 5.0:

        nice = 5.0

    elif normalized <= 7.5:

        nice = 7.5

    else:

        nice = 10.0


    ymax_nice = (
        nice * scale
    )


    if ymax_nice < 0.005:

        ymax_nice = 0.005


    return (
        0.0,
        ymax_nice
    )


# ============================================================
# MAIN PLOT LOOP
# ============================================================

for i, method in enumerate(
    method_names
):

    r2_min = R2_THRESHOLDS[
        method
    ]


    for j, structure in enumerate(
        structure_names
    ):

        ax = axes[
            i,
            j
        ]


        result = all_results[
            (
                structure,
                method
            )
        ]


        accepted = result.get(
            "accepted_batches",
            []
        )

        rejected = result.get(
            "rejected_batches",
            []
        )

        ensemble = result.get(
            "ensemble"
        )

        fit = result.get(
            "final_fit"
        )

        boot = result.get(
            "bootstrap"
        )


        # ====================================================
        # 1. REJECTED TRAJECTORIES
        # ====================================================

        for batch in rejected:

            t = np.asarray(
                batch["time_fs"],
                dtype=float
            )

            y = np.asarray(
                batch["population"],
                dtype=float
            )

            ax.plot(
                t,
                y,

                color="gray",

                linewidth=0.9,

                alpha=0.28,

                zorder=1,
            )


        # ====================================================
        # 2. ACCEPTED TRAJECTORIES
        # ====================================================

        for batch in accepted:

            t = np.asarray(
                batch["time_fs"],
                dtype=float
            )

            y = np.asarray(
                batch["population"],
                dtype=float
            )

            ax.plot(
                t,
                y,

                color="blue",

                linewidth=1.3,

                alpha=0.58,

                zorder=2,
            )


        # ====================================================
        # 3. ENSEMBLE FIT
        # ====================================================

        if (
            fit is not None
            and ensemble is not None
        ):

            t_grid = np.asarray(
                ensemble["time_fs"],
                dtype=float
            )


            tau_fs = fit[
                "tau_fs"
            ]

            beta = fit[
                "beta"
            ]


            fit_y = stretched_exp(
                t_grid,
                tau_fs,
                beta
            )


            tau_ps, err_ps = (
                get_tau_ps_and_error(
                    result
                )
            )


            # ------------------------------------------------
            # LEGEND TEXT
            # ------------------------------------------------

            if np.isfinite(
                err_ps
            ):

                label_text = (
                    f"$\\tau = "
                    f"{tau_ps:.1f} "
                    f"\\pm "
                    f"{err_ps:.1f}\\ \\mathrm{{ps}}$\n"
                    f"$R^2 \\geq "
                    f"{r2_min:.1f}$"
                )

            else:

                label_text = (
                    f"$\\tau = "
                    f"{tau_ps:.1f}\\ \\mathrm{{ps}}$\n"
                    f"$R^2 \\geq "
                    f"{r2_min:.1f}$"
                )


            # ------------------------------------------------
            # Main fitted curve
            # ------------------------------------------------

            ax.plot(
                t_grid,
                fit_y,

                color="red",

                linewidth=3.2,

                linestyle="-",

                label=label_text,

                zorder=5,
            )


            # =================================================
            # 4. BOOTSTRAP CI
            # =================================================

            if boot is not None:

                lower = np.asarray(
                    boot["fit_lower"],
                    dtype=float
                )

                upper = np.asarray(
                    boot["fit_upper"],
                    dtype=float
                )


                ax.fill_between(
                    t_grid,
                    lower,
                    upper,

                    color="red",

                    alpha=0.08,

                    zorder=3,
                )


                ax.plot(
                    t_grid,
                    lower,

                    color="red",

                    linestyle="--",

                    linewidth=1.3,

                    alpha=0.75,

                    zorder=4,
                )


                ax.plot(
                    t_grid,
                    upper,

                    color="red",

                    linestyle="--",

                    linewidth=1.3,

                    alpha=0.75,

                    zorder=4,
                )


        # ====================================================
        # 5. X AXIS
        # ====================================================

        ax.set_xlim(
            0,
            3000
        )

        ax.set_xticks(
            np.arange(
                0,
                3001,
                1000
            )
        )


        # ====================================================
        # 6. INDEPENDENT Y AXIS
        # ====================================================

        ymin, ymax = (
            determine_panel_ylim(
                result
            )
        )


        ax.set_ylim(
            ymin,
            ymax
        )


        ax.yaxis.set_major_locator(
            MaxNLocator(
                nbins=5,
                min_n_ticks=4
            )
        )


        # ----------------------------------------------------
        # Y number formatting
        # ----------------------------------------------------

        if ymax >= 0.1:

            y_format = "%.2f"

        elif ymax >= 0.01:

            y_format = "%.3f"

        elif ymax >= 0.001:

            y_format = "%.4f"

        else:

            y_format = "%.5f"


        ax.yaxis.set_major_formatter(
            FormatStrFormatter(
                y_format
            )
        )


        # ====================================================
        # 7. TICKS
        # ====================================================

        ax.tick_params(
            axis="both",

            labelsize=22,

            width=2.5,

            length=9,

            direction="out",

            pad=4,
        )


        ax.tick_params(
            axis="y",

            labelsize=20
        )


        # ====================================================
        # 8. SPINES
        # ====================================================

        for spine in ax.spines.values():

            spine.set_linewidth(
                2.5
            )


        ax.set_facecolor(
            "white"
        )


        # ====================================================
        # 9. LEGEND
        # ====================================================

        if fit is not None:

            legend = ax.legend(
                fontsize=19,

                loc="upper left",

                bbox_to_anchor=(
                    0.03,
                    0.97
                ),

                borderaxespad=0.0,

                frameon=True,

                fancybox=False,

                framealpha=0.92,

                edgecolor="black",

                handlelength=1.3,

                handletextpad=0.45,

                borderpad=0.40,

                labelspacing=0.28,

                columnspacing=0.5,
            )


        # ====================================================
        # 10. STRUCTURE TITLES
        #
        # NOT bold, larger, closer.
        # ====================================================

        if i == 0:

            title = structure


            if len(title) > 16:

                title = title.replace(
                    "-",
                    "-\n",
                    1
                )


            ax.set_title(
                title,

                fontsize=28,
                fontweight="normal",

                pad=15,
            )


        # ====================================================
        # 11. METHOD LABEL
        #
        # NOT bold, larger, closer.
        # ====================================================

        if j == 0:

            ax.set_ylabel(
                method,

                fontsize=28,

                fontweight="normal",

                labelpad=5,
            )


# ============================================================
# COMMON X AXIS LABEL
#
# NOT bold, larger, closer to the panels.
# ============================================================

fig.text(
    0.5,
    0.028,

    "Time (fs)",

    ha="center",
    va="center",

    fontsize=38,

    fontweight="normal",
)


# ============================================================
# COMMON Y AXIS LABEL
#
# NOT bold, larger, closer to the panels.
# ============================================================

fig.text(
    0.014,
    0.5,

    "Population (S$_0$)",

    ha="center",
    va="center",

    rotation="vertical",

    fontsize=38,

    fontweight="normal",
)


# ============================================================
# COMMON FIGURE TITLE
#
# NOT bold, larger, centered with equal margins.
# ============================================================

fig.suptitle(
    FIGURE_TITLE,

    fontsize=44,

    fontweight="normal",

    x=0.5,

    y=0.925,
)

# ============================================================
# LAYOUT
#
# Adjusted so the bigger title and labels have room,
# while panels remain large and tight.
#
# NOTE:
#   - tight_layout is called ONCE, AFTER suptitle,
#     so the title is taken into account in the layout.
#   - 'top' in rect leaves enough empty space above panels
#     for the large (fontsize=32) main title.
# ============================================================

# ------------------------------------------------------------
# 1) Common figure title FIRST
#    (so tight_layout knows about it)
# ------------------------------------------------------------

plt.tight_layout(
    rect=[
        0.045,        # left
        0.075,        # bottom
        0.995,        # right
        0.930         # top  (leaves room for large title)
    ],

    h_pad=0.75,

    w_pad=0.75,
)


# ------------------------------------------------------------
# 3) Force a draw so all positions / bboxes are finalized
# ------------------------------------------------------------

fig.canvas.draw()


# ============================================================
# SAVE
#
# bbox_inches="tight"  →  includes title, axis labels,
#                         ticks, legends
# pad_inches=0.25      →  extra safety margin around
#                         everything so nothing is clipped
# ============================================================

plt.savefig(
    FIGURE_OUT,

    dpi=600,

    bbox_inches="tight",

    pad_inches=0.25,

    facecolor="white",
)


print(
    f"Saved figure: {FIGURE_OUT}"
)